In [1]:
import glob
import numpy as np
import pandas as pd
import xarray as xr
from metpy.interpolate import log_interpolate_1d
from metpy.units import units
import matplotlib.pyplot as plt

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

def anomaly_calculation(model, clim):
    ano_list = []
    for i in range(1,13):
        ano_list.append(model.sel(time=model.time[model.time.dt.month.isin(i)]) - clim.sel(month=i))
    return xr.concat(ano_list, dim='time').sortby('time')

def calculate_anomaly_and_save_netcdf(model, var):
    ens = []
    ds = []
    for path in glob.glob('/sto4/data/Intermediate/VolMIP/pinatubo-full/' + model + '/' + var + '*fixed_dates.nc'):
        ens.append(path.split('/')[-1].split('_')[4])
        ds.append(xr.open_dataset(path))
    data = xr.concat(ds, dim='ens')
    data['ens'] = ens
    # open climatology and prepare it for anomaly calculation
    clim = xr.open_dataset(glob.glob('/sto4/data/Intermediate/VolMIP/piControl/' + model + '/' + var + '_clim*.nc')[0])
    clim['time'] = [1,2,3,4,5,6,7,8,9,10,11,12]
    clim = clim.rename({'time': 'month'})

    # select only the pressure levels that are available in the data
    data = data.sel(plev=clim.plev.values)
    # calculate anomalies
    data['ano'] = anomaly_calculation(data[var], clim[var])
    data.to_netcdf('/sto4/data/Intermediate/VolMIP/pinatubo-full/' + model + '/' + var + '_anomaly_' + model + '.nc')
    return 

In [2]:
models = ['CanESM5', 'GISS-E2-1-G','MPI-ESM1-2-LR', 'MIROC-ES2L', 'UKESM1-0-LL']

In [3]:
# for model in models: 
#     calculate_anomaly_and_save_netcdf(model, 'ta')
#     calculate_anomaly_and_save_netcdf(model, 'ua')

### Calculate neutral Climatology

In [4]:
def calculate_djf_mean(ds):
    ds_seasons = ds.resample(time='QE-FEB').mean()
    ds_djf = ds_seasons.sel(time=(ds_seasons['time.season'] == 'DJF'))
    return ds_djf

def calculate_tercile_flags(data, var):
    # calculate terciles to cluster indices into 3 categories: -1 (negative/cold), 0 (netural), 1 (positive/warm state)
    df = data[var].to_dataframe()
    df = df.reset_index()
    df = df.set_index('time')
    tertiles = df[var].quantile([1/3, 2/3]).to_list()
    tertiles = [-float('inf')] + tertiles + [float('inf')]

    df['tercile'] = pd.cut(df[var], bins=tertiles, labels=[-1,0,1])
    return df['tercile']

In [7]:
var = 'utendepfd'
for model in models[0:1]: 
    clim = xr.open_dataset(glob.glob('/sto4/data/Intermediate/VolMIP/piControl/' + model + '/' + var + '_EmonZ*.nc')[0])
    clim_djf = calculate_djf_mean(clim) 
    # nao = xr.open_dataset(glob.glob('/sto4/data/Intermediate/VolMIP/piControl/' + model + '/pi*'+ model + '*NAO_djf*.nc')[0])
    # df = calculate_tercile_flags(nao, 'zg').to_frame()
    # clim_djf.sel(time=df.where(df.tercile == 0).dropna().index).mean('time').to_netcdf('/sto4/data/Intermediate/VolMIP/piControl/' + model + '/' + 'naoneutral_ta_clim-djf_' + model + '_piControl.nc')
    # clim_djf.sel(time=df.where(df.tercile == -1).dropna().index).mean('time').to_netcdf('/sto4/data/Intermediate/VolMIP/piControl/' + model + '/' + 'naonegative_ta_clim-djf_' + model + '_piControl.nc')
    # clim_djf.sel(time=df.where(df.tercile == 1).dropna().index).mean('time').to_netcdf('/sto4/data/Intermediate/VolMIP/piControl/' + model + '/' + 'naopositive_ta_clim-djf_' + model + '_piControl.nc')
    clim_djf.mean('time').to_netcdf('/sto4/data/Intermediate/VolMIP/piControl/' + model + '/' + var + '_clim-djf_' + model + '_piControl.nc')

/tmp/ipykernel_3184458/16595285.py:3: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  clim = xr.open_dataset(glob.glob('/sto4/data/Intermediate/VolMIP/piControl/' + model + '/' + var + '_EmonZ*.nc')[0])
